In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import recall_score, precision_score
from sklearn.model_selection import GridSearchCV

In [2]:
data = pd.read_csv("Customer churn kaggle data.csv")

In [3]:
data = data.drop('customerID', axis = 1)
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors = "coerce")

In [4]:
x = data.drop('Churn', axis = 1)
y = data['Churn'].map({"Yes": 1, "No": 0})

### 🔹 1. Create NEW Feature

In [5]:
x['tenure_group'] = pd.cut(x['tenure'], bins = [0, 12, 24, 48, 72], labels = [0,1,2,3])

### 🔹 2. Create Interaction Feature

In [6]:
x['monthly_total_ratio'] = x['MonthlyCharges']/(x['TotalCharges'] + 1)

### 🔹 3. Sqrt Transform 

In [7]:
# The skewness of TotalCharges was 0.9 so we are going to transform it
x['TotalCharges'] = np.sqrt(x['TotalCharges'])

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [9]:
num_cols = x_train.select_dtypes(include = ['int64', 'float64']).columns
cat_cols = x_train.select_dtypes(include = 'object').columns

In [10]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = "mean")),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = "most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [11]:
preprocessor = ColumnTransformer([
    ('nums', num_pipeline, num_cols),
    ('cats', cat_pipeline, cat_cols)
])

In [12]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])

In [13]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear']
}

In [14]:
grid = GridSearchCV(pipeline, param_grid, cv = 5, scoring = "recall", n_jobs = -1)

In [15]:
grid.fit(x_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('nums',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'monthly_total_ratio'],
      dtype='object')),
                                                                        ('cats',
                                                                         Pipeline(steps=[('imputer',
                                             

In [22]:
best_model = grid.best_estimator_
print(grid.best_params_)

{'model__C': 0.1, 'model__penalty': 'l1', 'model__solver': 'liblinear'}


## best_model is not the actual model it contains best model + all the features so we need to fetch out our model and features from it

In [23]:
model = best_model.named_steps['model']
coefficients = model.coef_[0] # here we use this bcz coef_ is 2d array so we are selecting first row

# Get feature names after preprocessing and since we have older version of sklearn we won't be able to user get_feature_names_out()
feature_names = []

for name, transformer, cols in best_model.named_steps['preprocessing'].transformers_:
    # Columns which are not transformed
    if name == 'remainder': 
        continue

# In our case we are using transformer in pipeline so 
# [-1] gives last element that is encoder
# [1] gives OneHotEncoder()
    if hasattr(transformer, 'steps'):
        transformer = transformer.steps[-1][1]

# If transformer object is instance of the OneHotEncoder class, basically we are checking is oneHotEncoder has created any new columns or not
    if isinstance(transformer, OneHotEncoder):
        names = transformer.get_feature_names(cols)

# If oneHotEncoder is not used then that means no new columns were made like in imputer and standardScaler so directly put the features in names
    else:
        names = cols
    feature_names.extend(names)

# for name, coef in zip(feature_names, coefficients):
#     if abs(coef) < 0.05:
#         print(name, coef)


### We say that L1 regularization has already performed feature selection by shrinking weak feature coefficients to zero

In [18]:
y_pred = best_model.predict(x_test)

In [19]:
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Recall: 0.7994652406417112
Precision: 0.5033670033670034
